In [3]:
from dotenv import load_dotenv

load_dotenv()


True

# 2.4 Wedding Planner

This build uses one coordinator and four specialists.

```mermaid
graph TD
    U[Wedding request] --> C[Coordinator]
    C --> S[WeddingState]
    S --> F[Travel agent + flight MCP]
    S --> V[Venue agent + web search]
    S --> D[DJ agent + Chinook database]
    F --> C
    V --> C
    D --> C
    C --> O[Coordinated wedding plan]
```

The coordinator owns the vision and shared state. Each specialist receives only the fields it needs, completes one job, and returns a result for the coordinator to compile.

> Note: This lab includes retries for transient MCP failures and search limits. Tool errors are returned to the agent so it can adjust rather than crashing the whole run.

## Setup Tools


In [1]:
import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}

class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt+1}/{self.max_retries}): {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()

In [29]:
from typing import Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> dict[str, Any]:
    """Search the web with a strict limit and return compact evidence."""
    if search_number > max_search_number:
        return {"message": "Search limit reached. Summarize the findings already collected."}
    try:
        raw = tavily_client.search(query, max_results=2)
        return {
            "query": query,
            "results": [
                {
                    "title": item.get("title", ""),
                    "url": item.get("url", ""),
                    "content": item.get("content", "")[:350],
                }
                for item in raw.get("results", [])
            ],
        }
    except Exception as exc:
        return {"error": str(exc)}

In [4]:
from pathlib import Path
from langchain_community.utilities import SQLDatabase

database_path = Path.cwd() / "resources" / "Chinook.db"
db = SQLDatabase.from_uri(f"sqlite:///{database_path.as_posix()}")

@tool
def query_playlist_db(query: str) -> str:
    """Query the SQLite music database and return errors as tool data."""
    try:
        return db.run(query)
    except Exception as exc:
        return f"Error querying database: {exc}"

C:\Users\thinkpad\AppData\Local\Temp\ipykernel_6988\3341483583.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Create State

In [6]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents


In [18]:
from langchain.agents import create_agent

flight_search_tool = next(
    mcp_tool for mcp_tool in tools if mcp_tool.name == "search-flight"
)

@tool
async def flight_lookup(origin: str, destination: str) -> str:
    """Use the Kiwi flight MCP server for a round-trip economy search."""
    result = await flight_search_tool.ainvoke({
        "flyFrom": origin,
        "flyTo": destination,
        "departureDate": "15/06/2027",
        "returnDate": "22/06/2027",
        "adults": 1,
        "cabinClass": "M",
        "currency": "EUR",
        "locale": "en",
        "sort": "price",
    })
    return str(result)[:2500]

# Travel agent sees a small, stable interface while the wrapper uses the full MCP schema underneath.
travel_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[flight_lookup],
    system_prompt="""
    You are a travel agent. Use the flight lookup tool to find flights to and from the destination wedding.
    Return a concise shortlist ranked by price and duration. Do not ask follow-up questions.
    """
)

In [30]:
# Venue agent
venue_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location and capacity.
    Make no more than two web_search calls. Count every call and stop at the limit.
    Rank the best options by price, capacity, and reviews. Do not ask follow-up questions.
    Summarize the evidence you found, even when a source is incomplete.
    """
)

In [9]:
# Playlist agent
playlist_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## Main Coordinator


In [10]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


In [27]:
from langchain.agents import create_agent

coordinator = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator.

    Follow this checklist in order:
    1. Extract origin, destination, guest count, and genre from the request.
    2. Call update_state alone and wait for its result.
    3. Call search_flights, search_venues, and suggest_playlist exactly once each. These calls are required even if an earlier specialist returns an error.
    4. After all three specialist results are available, write one final wedding plan that clearly labels flights, venue, and playlist.

    Do not ask follow-up questions when all four facts are present. Do not stop after only one or two specialists. The final response must be text, not another tool call.
    """
)

## Test


In [31]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

In [13]:
from pprint import pprint

pprint(response)

{'destination': 'Paris',
 'genre': 'jazz',
 'guest_count': '100',
 'messages': [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre", additional_kwargs={}, response_metadata={}, id='4ad4ec40-39db-4b6f-ad9f-d9fc7ac7dec9'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 357, 'prompt_tokens': 336, 'total_tokens': 693, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DGpWpAAAUdxX1rcQUJfuLaPeLXFyi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cc94b-849d-7962-a708-8c55dfab6b0e-0', tool_calls=[{'name': 'update_state', 'args': {'origin': 'London', 'destinat

In [32]:
print("MESSAGE FLOW")
for message in response["messages"]:
    print(
        type(message).__name__,
        "name=", getattr(message, "name", None),
        "tool_calls=", [call.get("name") for call in getattr(message, "tool_calls", [])],
        "content_length=", len(str(getattr(message, "content", ""))),
    )

print("\nSTATE")
for key in ("origin", "destination", "guest_count", "genre"):
    print(key, "=", response.get(key))

MESSAGE FLOW
HumanMessage name= None tool_calls= [] content_length= 74
AIMessage name= None tool_calls= ['update_state'] content_length= 0
ToolMessage name= update_state tool_calls= [] content_length= 26
AIMessage name= None tool_calls= ['search_flights'] content_length= 0
ToolMessage name= search_flights tool_calls= [] content_length= 526
AIMessage name= None tool_calls= ['search_venues'] content_length= 0
ToolMessage name= search_venues tool_calls= [] content_length= 2430
AIMessage name= None tool_calls= ['suggest_playlist'] content_length= 0
ToolMessage name= suggest_playlist tool_calls= [] content_length= 1102
AIMessage name= None tool_calls= [] content_length= 2142

STATE
origin = London
destination = Paris
guest_count = 100
genre = jazz


## Take the architecture somewhere else: trip planner

The supervisor pattern is reusable. Only the state fields, specialist prompts, and coordinator tools change.

For a trip planner, the shared state is `origin`, `destination`, `traveler_count`, and `travel_style`. The coordinator still extracts facts, updates state, delegates, and compiles the answer.

In [23]:
from langchain.agents import AgentState, create_agent
from langchain.tools import ToolRuntime

class TripState(AgentState):
    origin: str
    destination: str
    traveler_count: str
    travel_style: str

lodging_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[web_search],
    system_prompt="You are a lodging specialist. Search for practical lodging options in the destination for the stated group and travel style. Return a concise shortlist.",
)

activity_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[web_search],
    system_prompt="You are a local activity specialist. Search for a short list of activities that match the destination and travel style. Return concise, source-backed suggestions.",
)

@tool
async def trip_flights(runtime: ToolRuntime) -> str:
    """Ask the flight specialist for a round-trip shortlist."""
    state = runtime.state
    result = await travel_agent.ainvoke({
        "messages": [HumanMessage(content=f"Find round-trip flight options from {state['origin']} to {state['destination']} for {state['traveler_count']} travelers.")]
    })
    return result["messages"][-1].content

@tool
def trip_lodging(runtime: ToolRuntime) -> str:
    """Ask the lodging specialist for options."""
    state = runtime.state
    result = lodging_agent.invoke({
        "messages": [HumanMessage(content=f"Find lodging in {state['destination']} for {state['traveler_count']} travelers with a {state['travel_style']} style.")]
    })
    return result["messages"][-1].content

@tool
def trip_activities(runtime: ToolRuntime) -> str:
    """Ask the activity specialist for an itinerary shortlist."""
    state = runtime.state
    result = activity_agent.invoke({
        "messages": [HumanMessage(content=f"Find activities in {state['destination']} for a {state['travel_style']} trip.")]
    })
    return result["messages"][-1].content

@tool
def update_trip_state(origin: str, destination: str, traveler_count: str, travel_style: str, runtime: ToolRuntime) -> Command:
    """Store the trip facts before calling the specialists."""
    return Command(update={
        "origin": origin,
        "destination": destination,
        "traveler_count": traveler_count,
        "travel_style": travel_style,
        "messages": [ToolMessage("Trip state updated", tool_call_id=runtime.tool_call_id)],
    })

trip_coordinator = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[trip_flights, trip_lodging, trip_activities, update_trip_state],
    state_schema=TripState,
    system_prompt="""
    You are a trip coordinator.
    First extract origin, destination, traveler count, and travel style from the request and update the state.
    After the state update returns, delegate flights, lodging, and activities to the specialists.
    Compile their results into one practical trip plan. Do not ask follow-up questions when the request already contains these four facts.
    """,
)

print("Trip planner built with shared state and three specialist tools.")

Trip planner built with shared state and three specialist tools.


## Community handoff

### Architecture

The coordinator owns the shared wedding vision and writes `origin`, `destination`, `guest_count`, and `genre` into `WeddingState`. The travel agent reads origin and destination and calls the flight MCP server. The venue agent reads destination and guest count and calls bounded web search. The DJ agent reads genre and queries Chinook. The coordinator compiles all three results.

### What each agent needed

- Travel: a compact wrapper around the large Kiwi MCP schema and a bounded result payload.
- Venue: a search counter, plain-text query rules, and compact title, URL, and excerpt evidence.
- DJ: schema discovery before SQL, plus retries for query mistakes.
- Coordinator: explicit state-update ordering and a required checklist for all specialists.

### Prompt rule learned

The coordinator initially stopped after returning venue results. The rule added afterward was: **call flights, venues, and playlist exactly once each, then write a final text synthesis; do not stop after partial specialist results.**

### Reusable lesson

The same supervisor and state pattern now builds a trip planner with flight, lodging, and activity specialists. Only the state fields, specialist prompts, and coordinator tools changed.

## LangSmith trace

The wedding test is tagged `WP`. Open LangSmith and filter runs by the `WP` tag to inspect the coordinator, specialist calls, MCP flight request, web searches, and database query as one trace tree.